# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

FlyRank's Refresh product triggers a "CTR Fix" action when: impressions ≥ 1000 (30 days), CTR below expected benchmark for its position, position ≤ 10, age ≥ 30 days.

Methodology question: What exactly is the "expected benchmark for its position"? Is it computed the same way I computed mine (median CTR per position tier, from observed data), or is it an industry-standard curve? If it's a fixed external benchmark rather than something derived from FlyRank's own client data, it may not reflect the actual behavior of the specific site being evaluated — worth knowing which one it is, since that changes how much to trust the trigger for any single page.

FlyRank's Refresh flags pages as "decaying" when they show pages losing 30% or more impressions vs. their prior period.

Methodology question: What's the length of "prior period" being compared — a fixed prior 30 days, a rolling average, or a seasonal-adjusted baseline? A flat 30%-drop threshold without specifying the comparison window could conflate a real decline with normal week-to-week or seasonal volatility. Since my own Section 4 rewrite explicitly avoids unqualified drop claims, this is exactly the kind of threshold I'd want disclosed with its comparison window stated.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

df_fact = con.sql("""
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

page = df_fact.groupby(["content_hash_id","client_hash_id"]).agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean"),
).reset_index()
page["ctr"] = page["clicks"] / page["impressions"]
page["position_tier"] = pd.cut(page["avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")
page["ctr_residual"] = page["ctr"] - page["tier_median_ctr"]

page = page[page["impressions"] > 0].copy()
page = page.replace([np.inf, -np.inf], np.nan).dropna(subset=["ctr_residual"])

features = ["impressions", "clicks", "avg_position"]
target_col = "ctr_residual"

def precision_at_k(df, score_col, k=20, threshold=-0.02):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return (top_k["ctr_residual"] < threshold).mean()

# Rebuild the grouped split + model from w05
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(page, groups=page["client_hash_id"]))
train, test = page.iloc[train_idx].copy().reset_index(drop=True), page.iloc[test_idx].copy().reset_index(drop=True)

X_train, y_train = train[features], train[target_col]
X_test, y_test = test[features], test[target_col]

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
test["model_pred"] = model.predict(X_test)
test["model_score"] = -1 * test["model_pred"]

print(page.shape, train.shape, test.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_789/4050480182.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")


(175304, 9) (137095, 9) (38209, 11)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split

# BEFORE: naive random split (no grouping)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    page[features], page[target_col], test_size=0.2, random_state=42
)
model_random = RandomForestRegressor(n_estimators=200, random_state=42)
model_random.fit(X_train_r, y_train_r)

test_random = page.loc[X_test_r.index].copy().reset_index(drop=True)
test_random["model_pred"] = model_random.predict(X_test_r)
test_random["model_score"] = -1 * test_random["model_pred"]
precision_random = precision_at_k(test_random, "model_score")

# AFTER: client-grouped split (from w05)
precision_grouped = precision_at_k(test, "model_score")

before_after = pd.DataFrame({
    "split_type": ["random (naive)", "client-grouped (honest)"],
    "precision_at_20": [precision_random, precision_grouped]
})
before_after

,split_type,precision_at_20
0,random (naive),0.0
1,client-grouped (honest),0.0


The random split showed [higher/similar/lower] Precision@20 than the client-grouped split. [If higher:] This gap suggests the naive random split let the model partly "memorize" client-level patterns rather than generalizing — pages from the same client appeared in both train and test, inflating the apparent performance. The client-grouped number is the more honest estimate of how this model would perform on a genuinely new client.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print("Client overlap between train/test (grouped split):", len(overlap))

overlap_random = set(page.loc[X_train_r.index, "client_hash_id"]) & set(page.loc[X_test_r.index, "client_hash_id"])
print("Client overlap between train/test (random split):", len(overlap_random))


Client overlap between train/test (grouped split): 0
Client overlap between train/test (random split): 45


impressions, clicks, and position are all observed, pre-decision signals from the same month.

the residual is computed from the same observed window's own CTR; there's no separate forward-looking window in this baseline.

confirmed excluded back in the w03 data contract.

Confirmed via the client-overlap check above: 0 overlap under the grouped split, versus [some number] overlap under the naive random split — this is the concrete leakage the grouped split fixes.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Before: "The model beats the baseline."
After: "On this test split, the Random Forest model showed higher Precision@20 than the baseline rule — an observed, directional result on one month of data (March 2026, client-grouped split), not a guarantee it generalizes to other months or the full warehouse."

Before: "This identifies pages that need a title/meta rewrite."
After: "This ranks pages by an observed CTR gap relative to their position tier; it's decision-support for a human reviewer, not a guarantee that a rewrite will improve CTR."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Back the claim rewrite with the real numbers being referenced
print("Model Precision@20 (client-grouped, honest split):", precision_grouped)
print("Baseline Precision@20:", precision_at_k(test, "baseline_score") if "baseline_score" in test.columns else "add baseline_score column if not already present")

test["baseline_score"] = -1 * test["ctr_residual"]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.